In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
from QRT_comp.phase2_qrt_challenge.scripts.utils import backtest_portfolio
import pandas as pd


In [ ]:
# Load the downloaded Yahoo Finance data
print('Loading top_5000_yf_data.pkl...')
df_historical = pd.read_pickle('../top_5000_yf_data.pkl')
print('Data loaded successfully.')

Loading top_5000_yf_data.pkl...


FileNotFoundError: [Errno 2] No such file or directory: 'top_5000_yf_data.pkl'

In [ ]:
# 1. Calculate Average Daily Volume (ADV) for trailing 60 days\n,
df_daily_volume = df_historical['Close'].mul(df_historical['Volume']).fillna(0)
df_adv_60 = df_daily_volume.rolling(window=60, min_periods=60).mean()
# 2. Price constraint
price_mask = df_historical['Close'] >= 1
# 3. ER (Efficiency Ratio) <= 0.25
# ER = absolute net change over 60 days / sum of absolute daily changes over 60 days\n",
net_change = df_historical['Close'].diff(60).abs()
sum_abs_daily_change = df_historical['Close'].diff().abs().rolling(window=60, min_periods=60).sum()
er = net_change / sum_abs_daily_change
er_mask = er <= 0.25
# 4. Volatility >= median\n",
# Volatility as 60-day rolling std of daily returns\n",
daily_ret = df_historical['Close'].pct_change()
vol_60 = daily_ret.rolling(window=60, min_periods=60).std()
median_vol = vol_60.median(axis=1)
# Compare each stock's volatility to the cross-sectional median for that day\n",
vol_mask = vol_60.ge(median_vol, axis=0)

In [ ]:
# Creating Universe based on all constraints
df_universe_5m = ((df_adv_60 >= 5_000_000)).astype(int)
print('Universe shape:', df_universe_5m.shape)

df_universe_5m.sum(axis=1).plot(title="Number of Tradable Stocks Over Time")


In [ ]:
# Saving 5M universe to stores folder
# Drop any duplicated columns
df_universe_5m = df_universe_5m.loc[:, ~df_universe_5m.columns.duplicated()]
df_universe_5m.to_parquet(os.path.join(DATA_DIR, 'universe_5m.parquet'), engine='pyarrow')
print('Saved universe_5m.parquet')


In [ ]:
# Calculating returns for each ticker every day
# Using Adj Close to account for dividends and stock splits
returns = df_historical['Adj Close'].pct_change(fill_method=None).fillna(0)

In [ ]:
# Saving returns to parquet file
# Drop any duplicated columns
returns = returns.loc[:, ~returns.columns.duplicated()]
returns.to_parquet(os.path.join(DATA_DIR, 'returns.parquet'), engine='pyarrow')
print('Saved returns.parquet')